# Case Study Assessment — NYC Airbnb Data Preprocessing
### Student Name:
### Date:

This notebook is your submission template. Fill in each section — do not delete the headers. Use markdown cells for your written justifications, right next to the code that supports them.

In [1]:
import pandas as pd
url = "https://raw.githubusercontent.com/erkansirin78/datasets/master/AB_NYC_2019.csv"
df = pd.read_csv(url)
df.head()

,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


---
## Part 1 — Data Understanding & Quality Audit

In [3]:
# TODO: shape, dtypes, missing value summary
print("Shape:", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing %": (df.isnull().mean() * 100).round(2)
})

print(missing_summary[missing_summary["Missing Count"] > 0])

Shape: (48895, 16)

Data types:
id                                  int64
name                               object
host_id                             int64
host_name                          object
neighbourhood_group                object
neighbourhood                      object
latitude                          float64
longitude                         float64
room_type                          object
price                               int64
minimum_nights                      int64
number_of_reviews                   int64
last_review                        object
reviews_per_month                 float64
calculated_host_listings_count      int64
availability_365                    int64
dtype: object

Missing values:
                   Missing Count  Missing %
name                          16       0.03
host_name                     21       0.04
last_review                10052      20.56
reviews_per_month          10052      20.56


In [4]:
# TODO: check for values that are technically present but don't make real-world sense
print("Price <= 0:", (df["price"] <= 0).sum())
print("Minimum nights > 365:", (df["minimum_nights"] > 365).sum())
print("Minimum nights > 1095:", (df["minimum_nights"] > 1095).sum())

print("Negative number of reviews:", (df["number_of_reviews"] < 0).sum())
print("Reviews per month < 0:", (df["reviews_per_month"] < 0).sum())
print("Availability outside 0-365:",
      ((df["availability_365"] < 0) | (df["availability_365"] > 365)).sum())

print("\nNumeric summary:")
display(df.describe())

Price <= 0: 11
Minimum nights > 365: 14
Minimum nights > 1095: 1
Negative number of reviews: 0
Reviews per month < 0: 0
Availability outside 0-365: 0

Numeric summary:


,id,host_id,latitude,longitude,price,minimum_nights,number_of_reviews,reviews_per_month,calculated_host_listings_count,availability_365
count,4.889500e+04,4.889500e+04,48895.000000,48895.000000,48895.000000,48895.000000,48895.000000,38843.000000,48895.000000,48895.000000
mean,1.901714e+07,6.762001e+07,40.728949,-73.952170,152.720687,7.029962,23.274466,1.373221,7.143982,112.781327
std,1.098311e+07,7.861097e+07,0.054530,0.046157,240.154170,20.510550,44.550582,1.680442,32.952519,131.622289
min,2.539000e+03,2.438000e+03,40.499790,-74.244420,0.000000,1.000000,0.000000,0.010000,1.000000,0.000000
25%,9.471945e+06,7.822033e+06,40.690100,-73.983070,69.000000,1.000000,1.000000,0.190000,1.000000,0.000000
50%,1.967728e+07,3.079382e+07,40.723070,-73.955680,106.000000,3.000000,5.000000,0.720000,1.000000,45.000000
75%,2.915218e+07,1.074344e+08,40.763115,-73.936275,175.000000,5.000000,24.000000,2.020000,2.000000,227.000000
max,3.648724e+07,2.743213e+08,40.913060,-73.712990,10000.000000,1250.000000,629.000000,58.500000,327.000000,365.000000


In [5]:
# TODO: duplicate check
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


**Written notes — what did you find, and what looks suspicious?**

The dataset contains 48,895 rows and 16 columns. The data contains numerical, categorical and date-related variables.

The main missing values occur in name, host_name, last_review and reviews_per_month. The last_review and reviews_per_month columns each contain 10,052 missing values.

The dataset also contains suspicious values. Price contains zero-dollar listings, which do not represent a valid price for a price-prediction task. minimum_nights has extreme values, including values greater than 365 days and a maximum of 1,250 days. These values are unusual for a short-term rental listing and should be investigated.

The maximum price is $10,000 per night. Although this is an extreme statistical outlier, it may represent a genuine luxury listing rather than a data-entry error, so it should not automatically be deleted.

The dataset has no duplicated complete rows.

---
## Part 2 — Missing Value Diagnosis & Treatment

In [6]:
# TODO: investigate missingness pattern(s) across columns
missing_summary = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percentage": (df.isnull().mean() * 100).round(2)
})

display(
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_count", ascending=False)
)

,missing_count,missing_percentage
reviews_per_month,10052,20.56
last_review,10052,20.56
host_name,21,0.04
name,16,0.03


In [7]:


review_missing = df["reviews_per_month"].isna()

print(pd.crosstab(
    review_missing,
    df["number_of_reviews"].eq(0),
    normalize="index"
))

print("\nRows with missing reviews_per_month:")
display(
    df.loc[review_missing, ["number_of_reviews", "last_review",
                            "reviews_per_month"]].head(10)
)

number_of_reviews  False  True 
reviews_per_month              
False                1.0    0.0
True                 0.0    1.0

Rows with missing reviews_per_month:


,number_of_reviews,last_review,reviews_per_month
2,0,NaN,NaN
19,0,NaN,NaN
26,0,NaN,NaN
36,0,NaN,NaN
38,0,NaN,NaN
193,0,NaN,NaN
204,0,NaN,NaN
260,0,NaN,NaN
265,0,NaN,NaN
267,0,NaN,NaN


In [8]:


for col in ["name", "host_name"]:
    print(f"\nMissingness of {col} by room type:")
    print(
        df.groupby("room_type")[col]
        .apply(lambda x: x.isna().mean())
        .round(4)
    )


Missingness of name by room type:
room_type
Entire home/apt    0.0003
Private room       0.0004
Shared room        0.0009
Name: name, dtype: float64

Missingness of host_name by room type:
room_type
Entire home/apt    0.0004
Private room       0.0005
Shared room        0.0000
Name: host_name, dtype: float64


**Written justification — MCAR / MAR / MNAR classification per column, and your reasoning:**

The missing values occur in name, host_name, last_review, and reviews_per_month.

name — approximately MCAR: Only a very small number of values are missing

host_name — approximately MCAR:

last_review — MAR:

reviews_per_month — MAR: Its missingness is also strongly associated with number_of_reviews = 0.

Overall, the strongest missingness pattern is in last_review and reviews_per_month, where the missing values are explained by the observed number_of_reviews column rather than occurring randomly.


In [9]:
# TODO: apply your chosen treatment(s)
df_clean = df.copy()

# Missing listing/host names
df_clean["name"] = df_clean["name"].fillna("Unknown listing")
df_clean["host_name"] = df_clean["host_name"].fillna("Unknown host")

# For listings with zero reviews:
# no review activity means 0 reviews per month
df_clean["reviews_per_month"] = df_clean["reviews_per_month"].fillna(0)

# last_review is not useful for predicting the initial price
# and its missingness has a meaningful interpretation.
# We will remove it from the modeling features later.

print(df_clean.isnull().sum())

id                                    0
name                                  0
host_id                               0
host_name                             0
neighbourhood_group                   0
neighbourhood                         0
latitude                              0
longitude                             0
room_type                             0
price                                 0
minimum_nights                        0
number_of_reviews                     0
last_review                       10052
reviews_per_month                     0
calculated_host_listings_count        0
availability_365                      0
dtype: int64


**Written justification — why this treatment for each column, and what you'd risk with `dropna()` instead:**

I replaced missing name and host_name values with explicit "Unknown" categories rather than deleting those rows. Only a very small number of records are affected, and deleting them would remove otherwise usable listings.

I replaced missing reviews_per_month with 0 because the missingness corresponds to listings with no reviews. In this situation, the missing value represents the absence of review activity rather than an unknown numerical measurement.

I did not attempt to invent values for last_review. A missing last review date has a meaningful interpretation for listings with no reviews, and last_review will not be used as a predictive feature.

Using dropna() on the entire dataset would unnecessarily discard around 10,000 listings because of the review-related missing values. It would also remove listings with missing names or host names even though those fields are not important for price prediction. Therefore, complete-case deletion would reduce the training data and could introduce selection bias.

---
## Part 3 — Outlier Detection & Treatment

In [10]:
# TODO: detect outliers in at least two numeric columns
import numpy as np

def iqr_outliers(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    mask = (data[column] < lower) | (data[column] > upper)

    return {
        "Q1": Q1,
        "Q3": Q3,
        "IQR": IQR,
        "lower_bound": lower,
        "upper_bound": upper,
        "outlier_count": mask.sum()
    }

price_outliers = iqr_outliers(df_clean, "price")
minimum_outliers = iqr_outliers(df_clean, "minimum_nights")

print("Price:")
print(price_outliers)

print("\nMinimum nights:")
print(minimum_outliers)

Price:
{'Q1': np.float64(69.0), 'Q3': np.float64(175.0), 'IQR': np.float64(106.0), 'lower_bound': np.float64(-90.0), 'upper_bound': np.float64(334.0), 'outlier_count': np.int64(2972)}

Minimum nights:
{'Q1': np.float64(1.0), 'Q3': np.float64(5.0), 'IQR': np.float64(4.0), 'lower_bound': np.float64(-5.0), 'upper_bound': np.float64(11.0), 'outlier_count': np.int64(6607)}


In [11]:

display(
    df_clean.nlargest(
        20, "price"
    )[[
        "price",
        "room_type",
        "neighbourhood_group",
        "neighbourhood",
        "minimum_nights",
        "number_of_reviews"
    ]]
)

,price,room_type,neighbourhood_group,neighbourhood,minimum_nights,number_of_reviews
9151,10000,Private room,Queens,Astoria,100,2
17692,10000,Entire home/apt,Brooklyn,Greenpoint,5,5
29238,10000,Entire home/apt,Manhattan,Upper West Side,30,0
6530,9999,Entire home/apt,Manhattan,East Harlem,5,1
12342,9999,Private room,Manhattan,Lower East Side,99,6
40433,9999,Entire home/apt,Manhattan,Lower East Side,30,0
30268,8500,Entire home/apt,Manhattan,Tribeca,30,2
4377,8000,Entire home/apt,Brooklyn,Clinton Hill,1,1
29662,7703,Entire home/apt,Manhattan,Upper East Side,1,0
42523,7500,Entire home/apt,Manhattan,Battery Park City,1,0


In [12]:
display(
    df_clean.nlargest(
        20, "minimum_nights"
    )[[
        "minimum_nights",
        "price",
        "room_type",
        "neighbourhood_group",
        "number_of_reviews",
        "availability_365"
    ]]
)

,minimum_nights,price,room_type,neighbourhood_group,number_of_reviews,availability_365
5767,1250,180,Entire home/apt,Manhattan,2,365
2854,1000,400,Entire home/apt,Manhattan,0,362
13404,999,99,Entire home/apt,Manhattan,2,42
26341,999,79,Private room,Brooklyn,24,249
38664,999,110,Shared room,Manhattan,0,365
7355,500,134,Entire home/apt,Queens,30,90
8014,500,75,Entire home/apt,Manhattan,0,362
11193,500,50,Private room,Brooklyn,10,365
14285,500,45,Private room,Brooklyn,0,358
47620,500,140,Entire home/apt,Brooklyn,0,331


In [13]:
print("Zero-price listings:", (df_clean["price"] == 0).sum())

display(
    df_clean[df_clean["price"] == 0][[
        "price",
        "room_type",
        "neighbourhood_group",
        "minimum_nights",
        "number_of_reviews",
        "availability_365"
    ]]
)

Zero-price listings: 11


,price,room_type,neighbourhood_group,minimum_nights,number_of_reviews,availability_365
23161,0,Private room,Brooklyn,4,1,28
25433,0,Private room,Bronx,2,55,127
25634,0,Private room,Brooklyn,2,16,0
25753,0,Private room,Brooklyn,2,12,0
25778,0,Entire home/apt,Brooklyn,5,3,73
25794,0,Private room,Brooklyn,1,93,176
25795,0,Private room,Brooklyn,1,95,232
25796,0,Private room,Brooklyn,1,95,222
26259,0,Entire home/apt,Manhattan,3,0,0
26841,0,Shared room,Brooklyn,30,2,333


**Written justification — for each outlier group, is it an error or a genuine listing? What evidence supports your call
The IQR method was used because price and minimum_nights are strongly right-skewed. A z-score method assumes a distribution closer to normality and would be less appropriate for these variables.

The zero-price listings are treated as data errors for this task. A nightly Airbnb price of $0 cannot be used as a valid target for a model predicting paid nightly prices, so these rows should be removed.

The $10,000 price values are statistical outliers but should not automatically be removed. A very expensive Manhattan entire-home listing could be a genuine luxury property. The room type, neighbourhood and other listing characteristics should therefore be considered before calling such a value an error. I retain genuine high-price listings rather than imposing an arbitrary maximum price.

minimum_nights values above 365 are suspicious because they require a minimum stay longer than a full year and the dataset is intended to represent short-term rental listings. These observations should be treated as invalid/extreme duration values rather than deleting every statistical outlier based only on the IQR rule.

The important distinction is that a statistical outlier is not automatically a data error. Removing every expensive listing could eliminate legitimate Manhattan luxury properties and bias the price model toward cheaper listings.

In [14]:
# TODO: apply treatment consistent with your judgment above

df_clean = df_clean[df_clean["price"] > 0].copy()
df_clean.loc[
    df_clean["minimum_nights"] > 365,
    "minimum_nights"
] = np.nan

print("Rows after removing zero-price listings:", len(df_clean))
print("Invalid minimum-night values remaining:",
      (df_clean["minimum_nights"] > 365).sum())

Rows after removing zero-price listings: 48884
Invalid minimum-night values remaining: 0


---
## Part 4 — Feature Engineering & Encoding

In [15]:
# TODO: encode categorical columns appropriately

categorical_columns = [
    "neighbourhood_group",
    "neighbourhood",
    "room_type"
]

for col in categorical_columns:
    print(f"\n{col}:")
    print(df_clean[col].unique())


neighbourhood_group:
['Brooklyn' 'Manhattan' 'Queens' 'Staten Island' 'Bronx']

neighbourhood:
['Kensington' 'Midtown' 'Harlem' 'Clinton Hill' 'East Harlem'
 'Murray Hill' 'Bedford-Stuyvesant' "Hell's Kitchen" 'Upper West Side'
 'Chinatown' 'South Slope' 'West Village' 'Williamsburg' 'Fort Greene'
 'Chelsea' 'Crown Heights' 'Park Slope' 'Windsor Terrace' 'Inwood'
 'East Village' 'Greenpoint' 'Bushwick' 'Flatbush' 'Lower East Side'
 'Prospect-Lefferts Gardens' 'Long Island City' 'Kips Bay' 'SoHo'
 'Upper East Side' 'Prospect Heights' 'Washington Heights' 'Woodside'
 'Brooklyn Heights' 'Carroll Gardens' 'Gowanus' 'Flatlands' 'Cobble Hill'
 'Flushing' 'Boerum Hill' 'Sunnyside' 'DUMBO' 'St. George' 'Highbridge'
 'Financial District' 'Ridgewood' 'Morningside Heights' 'Jamaica'
 'Middle Village' 'NoHo' 'Ditmars Steinway' 'Flatiron District'
 'Roosevelt Island' 'Greenwich Village' 'Little Italy' 'East Flatbush'
 'Tompkinsville' 'Astoria' 'Clason Point' 'Eastchester' 'Kingsbridge'
 'Two Bridg

In [16]:
# TODO: engineer at least two new features
# Feature 1: whether the listing is an entire home/apartment
df_clean["is_entire_home"] = (
    df_clean["room_type"] == "Entire home/apt"
).astype(int)

# Feature 2: whether the listing is in Manhattan
df_clean["is_manhattan"] = (
    df_clean["neighbourhood_group"] == "Manhattan"
).astype(int)

# Feature 3: log transformation of minimum nights
# Useful because minimum_nights is highly right-skewed
df_clean["log_minimum_nights"] = np.log1p(
    df_clean["minimum_nights"]
)

display(
    df_clean[
        [
            "room_type",
            "neighbourhood_group",
            "minimum_nights",
            "is_entire_home",
            "is_manhattan",
            "log_minimum_nights"
        ]
    ].head()
)

,room_type,neighbourhood_group,minimum_nights,is_entire_home,is_manhattan,log_minimum_nights
0,Private room,Brooklyn,1.0,0,0,0.693147
1,Entire home/apt,Manhattan,1.0,1,1,0.693147
2,Private room,Manhattan,3.0,0,1,1.386294
3,Entire home/apt,Brooklyn,1.0,1,0,0.693147
4,Entire home/apt,Manhattan,10.0,1,1,2.397895


**Written justification — why these features, and which column(s) should NOT be used to predict price, and why:**

I engineered is_entire_home because an entire home/apartment normally provides more space and privacy than a private or shared room, which can influence the nightly price.

I engineered is_manhattan because Manhattan is a major geographic factor in NYC accommodation pricing. A model can also learn this information from neighbourhood_group, but the explicit binary feature provides a simple interpretable representation of the geographic effect.

I also created log_minimum_nights because minimum_nights is highly right-skewed. The logarithmic transformation reduces the influence of very large values while preserving the information contained in the feature.

The following raw columns should not be used as predictive features for a new listing: id, host_id, name and host_name because they are identifiers or free-text labels rather than stable generalizable pricing factors.

More importantly, review-related variables such as number_of_reviews, last_review and reviews_per_month, as well as availability_365, contain information that becomes available after a listing has been active. Using them to predict the initial price of a listing would create temporal leakage because that information may not exist when the price prediction is actually required.

Therefore, I exclude these post-launch variables from the price-prediction model.

---
## Part 5 — Build a Reusable Preprocessing Pipeline

In [18]:
# TODO: train/test split (before fitting anything)
from sklearn.model_selection import train_test_split

drop_columns = [
    "id",
    "name",
    "host_id",
    "host_name",
    "last_review",
    "reviews_per_month",
    "number_of_reviews",
    "availability_365"
]
X = df_clean.drop(columns=["price"] + drop_columns)
y = df_clean["price"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training shape:", X_train.shape)
print("Testing shape:", X_test.shape)

Training shape: (39107, 10)
Testing shape: (9777, 10)


In [19]:
# TODO: build Pipeline combining your cleaning, imputation, encoding, scaling steps
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Numerical features
numeric_features = [
    "latitude",
    "longitude",
    "minimum_nights",
    "is_entire_home",
    "is_manhattan",
    "log_minimum_nights",
    "calculated_host_listings_count"
]

# Categorical features
categorical_features = [
    "neighbourhood_group",
    "neighbourhood",
    "room_type"
]

# Numerical preprocessing
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

# Categorical preprocessing
categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False
        ))
    ]
)

# Combine preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features)
    ]
)

# Complete preprocessing pipeline
preprocessing_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor)
    ]
)

# Fit ONLY on training data
X_train_processed = preprocessing_pipeline.fit_transform(X_train)

# Transform test data using the already-fitted pipeline
X_test_processed = preprocessing_pipeline.transform(X_test)

print("Processed training shape:", X_train_processed.shape)
print("Processed testing shape:", X_test_processed.shape)

Processed training shape: (39107, 233)
Processed testing shape: (9777, 233)


---
## Part 6 — Written Reflection

1. The largest preprocessing decision was handling the extreme and invalid values in price and minimum_nights. In particular, removing zero-price listings prevents the model from learning that a valid Airbnb can have a free nightly price, while retaining genuine expensive listings prevents the model from becoming biased toward ordinary low- and mid-priced properties. The impact can be evaluated by comparing model performance before and after preprocessing using the same train/test split.
2. If StayScope Analytics began receiving live listing data, the preprocessing logic would need to be applied consistently to incoming records, while all fitted statistics such as imputation medians and scaling parameters would remain those learned from the training data. The pipeline would also need monitoring for new categories, distribution changes and new types of missing or invalid values.
3. I would not delete every row with a missing or unusual value because missingness can contain meaningful information and statistical outliers can represent genuine Airbnb listings. Deleting all such rows would unnecessarily reduce the dataset and could introduce bias, whereas targeted treatment allows valid information to be retained while correcting genuinely invalid observations.